# 🎬 Estúdio — rodar no Google Colab

1. **Ambiente de execução → Alterar tipo → GPU T4.**
2. (Opcional) Em 🔑 **Secrets**, crie `HF_TOKEN` com um token *write* do Hugging Face — só é preciso para publicar o Space.
3. Rode as células em ordem. A célula 3 mostra um link `https://….gradio.live`: abra ele, ou cole na página `web/index.html`.

In [ ]:
# 1. Confere a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Baixa o código e instala as dependências (~3 min)
RAMO = "estudio-video"
!rm -rf Jarvis && git clone -q -b {RAMO} --depth 1 https://github.com/gutodrac-spec/Jarvis.git
%cd Jarvis/estudio
!pip install -q -r requirements.txt

In [ ]:
# 3. Sobe a API + interface com link público.
#    Na primeira geração ele baixa o modelo (~15 GB, alguns minutos).
!ESTUDIO_SHARE=1 python app.py

## Publicar como Hugging Face Space (opcional)

Cria (ou atualiza) um Space com este app. O Space grátis só tem CPU, que é lento demais para vídeo:
para gerar lá, escolha uma GPU nas configurações do Space ou use **ZeroGPU** (conta PRO).
Pare a célula 3 antes de rodar esta.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi

NOME_DO_SPACE = "estudio"  # vira https://huggingface.co/spaces/<seu-usuario>/estudio

api = HfApi(token=userdata.get("HF_TOKEN"))
repo = f"{api.whoami()['name']}/{NOME_DO_SPACE}"
api.create_repo(repo, repo_type="space", space_sdk="gradio", exist_ok=True)

with open("/tmp/README.md", "w") as f:
    f.write("---\ntitle: Estúdio\nemoji: 🎬\nsdk: gradio\napp_file: app.py\npinned: false\n---\n\n"
            "Texto ou imagem para vídeo com presets de câmera.\n")
api.upload_folder(repo_id=repo, repo_type="space", folder_path=".",
                  allow_patterns=["app.py", "presets.py", "requirements.txt"])
api.upload_file(repo_id=repo, repo_type="space", path_or_fileobj="/tmp/README.md", path_in_repo="README.md")
print("Space:", f"https://huggingface.co/spaces/{repo}")